# Colab SSH Setup

Run **Cell 1** once at the start of every Colab session.
It will print a hostname — paste it into your local `~/.ssh/config` under `Host colab`, then connect from VSCode.

Cells 2 and 3 are run **once per session from the VSCode terminal** after connecting.

In [ ]:
# ── Cell 1: SSH tunnel setup ─────────────────────────────────────────────────
# Run this every session. Copy the printed HostName into ~/.ssh/config on your Mac.

import subprocess, re, time, os
from pathlib import Path

PUBLIC_KEY = 'ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAIEnt50s+Sx/SVrypSYXQSdTASIW0oYjwSRxwG+c4Tfo8 mihirkul01@gmail.com'

# Install SSH server
os.system('apt-get install -qq -y openssh-server > /dev/null 2>&1')
os.makedirs('/root/.ssh', exist_ok=True)
Path('/root/.ssh/authorized_keys').write_text(PUBLIC_KEY + '\n')
os.chmod('/root/.ssh/authorized_keys', 0o600)
os.makedirs('/var/run/sshd', exist_ok=True)
with open('/etc/ssh/sshd_config', 'a') as f:
    f.write('\nPermitRootLogin yes\n')
os.system('service ssh restart > /dev/null 2>&1')

# Install cloudflared
os.system('wget -qO /tmp/cloudflared.deb https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb')
os.system('dpkg -i -q /tmp/cloudflared.deb > /dev/null 2>&1')

# Start tunnel
proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'ssh://localhost:22'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)

print('Starting tunnel', end='', flush=True)
hostname = None
for _ in range(60):
    line = proc.stderr.readline().decode()
    m = re.search(r'https://([a-zA-Z0-9-]+\.trycloudflare\.com)', line)
    if m:
        hostname = m.group(1)
        break
    print('.', end='', flush=True)
    time.sleep(1)

if hostname:
    print(f'\n\nTunnel ready! Update ~/.ssh/config on your Mac:\n')
    print(f'    HostName {hostname}')
    print(f'\nThen in VSCode: Cmd+Shift+P → Remote-SSH: Connect to Host → colab')
else:
    print('\nFailed to get tunnel URL. Re-run this cell.')

---
## After connecting from VSCode

Open VSCode's integrated terminal (the one connected to Colab) and run these two blocks once per session.

### First: clone repo + install deps
```bash
git clone --branch adaptive-attention https://github.com/Miwi343/Neural_Image_Caption_Generator.git ~/repo
cd ~/repo
pip install -q -r requirements.txt
```

### Second: download Flickr8k (~2–3 min)
```bash
mkdir -p ~/.kaggle
# paste your kaggle.json content below, or copy the file:
echo '{"username":"YOUR_USERNAME","key":"YOUR_KEY"}' > ~/.kaggle/kaggle.json
chmod 600 ~/.kaggle/kaggle.json

pip install -q kaggle
kaggle datasets download adityajn105/flickr8k --path ~/repo/data/flickr8k --unzip
```

### Then train:
```bash
cd ~/repo
python train_adaptive.py
```

### Before ending your session — push any code changes:
```bash
cd ~/repo
git add -p   # review what changed
git commit -m 'your message'
git push
```

> Checkpoints will be lost when the session ends unless you copy them to Drive:
> `cp -r ~/repo/checkpoints /content/drive/MyDrive/Neural_Image_Caption_Generator/`